[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C14_DL_Theory_Data_Course/05_generative_media_eval/05_generative_media_eval.ipynb)

# 05 · 生成媒体评测（用 numpy 从零算）

目标：在玩具高斯特征上**从零实现** FID(含矩阵平方根)、IS(含 KL)、improved precision-recall(k-NN 流形), 用 `assert` 钉死每条性质, 并构造反例看清各指标量什么、能被怎样骗。

路线：矩阵平方根 → FID(自反性+敏感性) → IS(奖励保真+多样, 两种失败) → k-NN 流形 → precision-recall(分别诊断坍塌/跑偏) → 保真-多样权衡 → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊(真实图像 patch 特征 FID)。

> 心智模型：**保真(单样本像不像真) 与 多样(覆不覆盖真实全部模式) 是两个独立维度**; FID 把它们混成一个数, precision-recall 把它们掰开。

## 1 · 矩阵平方根：FID 的唯一技术难点

FID 需要 `(Σ1·Σ2)^½` —— 满足 `S²=M` 的矩阵(不是逐元素开方)。对**对称半正定**矩阵, 用特征分解: `M=VΛVᵀ -> M^½ = V·√Λ·Vᵀ`。

数值细节: 先**对称化**(浮点误差), 再 **clip 负特征值**(否则 sqrt(负)=nan)。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def matrix_sqrt(M):
    '''对称半正定矩阵的平方根: M=VΛVᵀ -> V√ΛVᵀ。含对称化与负特征值 clip。'''
    M = (M + M.T) / 2.0                       # 对称化(消浮点不对称)
    w, v = np.linalg.eigh(M)                  # 对称矩阵特征分解
    w = np.clip(w, 0, None)                   # clip 微小负特征值
    return (v * np.sqrt(w)) @ v.T

# 验证 S² == M
A = rng.standard_normal((6, 6))
M = A @ A.T                                   # 对称半正定
S = matrix_sqrt(M)
print('||S @ S - M|| =', np.linalg.norm(S @ S - M))
assert np.allclose(S @ S, M, atol=1e-8), '矩阵平方根: S² 应等于 M'
assert np.allclose(S, S.T, atol=1e-8), '结果应对称'
print('✅ matrix_sqrt 正确: S² = M (特征分解法)')

## 2 · 从零算 FID

`FID = ||μ1-μ2||² + Tr(Σ1 + Σ2 - 2(Σ1Σ2)^½)`。第一项=均值差(保真), 第二项=协方差差(多样/结构)。

自检: **FID(X,X)=0**(自反性); 均值偏移、方差收缩都让 FID **增大**(对保真和多样都敏感)。

In [ ]:
def fid(X, Y):
    '''X(n,d) 真实特征, Y(m,d) 生成特征 -> FID。'''
    mu1, mu2 = X.mean(0), Y.mean(0)
    s1 = np.cov(X, rowvar=False)
    s2 = np.cov(Y, rowvar=False)
    diff = mu1 - mu2
    covmean = matrix_sqrt(s1 @ s2)
    return float(diff @ diff + np.trace(s1 + s2 - 2 * covmean))

d = 8
real = rng.standard_normal((600, d))
same = rng.standard_normal((600, d))          # 同分布(另采一批)
shifted = rng.standard_normal((600, d)) + 1.0  # 均值偏移(保真↓)
narrow = rng.standard_normal((600, d)) * 0.3   # 方差收缩(多样↓, 像坍塌)

print(f'FID(real, real 自己)   = {fid(real, real):.4f}')
print(f'FID(real, 同分布另采)  = {fid(real, same):.4f}')
print(f'FID(real, 均值偏移)    = {fid(real, shifted):.4f}')
print(f'FID(real, 方差收缩)    = {fid(real, narrow):.4f}')
assert abs(fid(real, real)) < 1e-6, 'FID(X,X) 必须=0(自反性)'
assert fid(real, same) < 1.0, '同分布 FID 应很小'
assert fid(real, shifted) > fid(real, same), '均值偏移(保真↓)推高 FID'
assert fid(real, narrow) > fid(real, same), '方差收缩(多样↓)也推高 FID'
print('✅ FID 从零实现正确: 自反性=0, 对保真↓和多样↓都敏感(后者正是 model collapse 的指纹)')

## 3 · 从零算 Inception Score

`IS = exp(E_x KL(p(y|x) || p(y)))`, 其中 `p(y)=mean_x p(y|x)`(边际)。**不需要真实图**。

奖励两件事同时成立: 每图分类**置信**(p(y|x) 尖锐=保真) + 整体类别**均匀**(p(y) 均匀=多样)。任一不满足 IS 都→1。

In [ ]:
def inception_score(probs):
    '''probs(N,C) = 每个生成样本的条件类别概率 p(y|x)。'''
    p_y = probs.mean(0)                       # 边际 p(y)
    kl = np.sum(probs * (np.log(probs + 1e-12) - np.log(p_y + 1e-12)), axis=1)
    return float(np.exp(kl.mean()))

C, N = 5, 1000
labels = rng.integers(0, C, N)
# 好: 每图置信(0.92) + 类别均匀
good = np.full((N, C), 0.02); good[np.arange(N), labels] = 0.92
# 坏1: 每图不置信(均匀 p(y|x))
unconfident = np.full((N, C), 1.0 / C)
# 坏2: 置信但全是同一类(无多样)
no_diversity = np.full((N, C), 0.02); no_diversity[:, 0] = 0.92

print(f'IS 好(置信+多样)   = {inception_score(good):.3f}  (上界≈类别数 {C})')
print(f'IS 不置信         = {inception_score(unconfident):.3f}  (≈1, 最差)')
print(f'IS 无多样(全一类) = {inception_score(no_diversity):.3f}  (≈1, 虽置信)')
assert inception_score(good) > 2.5, '置信+多样 -> IS 高'
assert abs(inception_score(unconfident) - 1.0) < 0.05, '不置信 -> IS≈1'
assert inception_score(no_diversity) < 1.1, '无多样 -> IS≈1(即便每图置信)'
print('✅ IS 从零实现: 奖励「每图置信(保真)+整体均匀(多样)」, 缺一则≈1')

## 4 · IS 的缺陷：不看真实图、易被钻空子

IS **不看真实分布** -> 一组与真实毫无关系、但清晰且类别均匀的图也能拿高 IS。
演示: 「钻空子」生成器 = 每类只生成 1 张完美图、复制很多份 -> IS 高, 但实际多样性(类内)为零。

In [ ]:
# 钻空子: 每个类别只有 1 个「原型」(超置信), 大量复制
C, N = 5, 1000
gamed = np.full((N, C), 0.005)
for i in range(N):
    gamed[i, i % C] = 0.98               # 轮流每类, 每张都超置信
print(f'IS 钻空子(每类1原型复制) = {inception_score(gamed):.3f}  <- 很高!')
# 但这组「图」类内零多样性(每类都一模一样). IS 看不出, 因为它只看类别分布
assert inception_score(gamed) > 4.0, 'IS 被钻空子刷到接近上界'
# 对照: FID 需要真实图, 能发现「生成分布≠真实分布」. IS 做不到这点
print('教训: IS 高 ≠ 真的好. 它不看真实分布、只看类别置信与均匀, 易被针对性优化(Goodhart)')
print('✅ IS 缺陷复现: 不看真实图 -> 清晰且类别均匀的「假多样」也能高分 -> 需 FID/PR/人评兜底')

## 5 · k-NN 流形 + improved precision-recall

用 k-NN 估计每个分布的**流形**(每点到第k近邻的球的并集)。
- **Precision(保真)** = 生成点落在**真实流形**内的比例;
- **Recall(多样)** = 真实点落在**生成流形**内的比例。

纯 numpy 算欧氏距离。这把保真和多样**掰成两个数**。

In [ ]:
def pairwise_dist(A, B):
    '''纯 numpy 欧氏距离矩阵 (len A, len B)。'''
    a2 = np.sum(A ** 2, axis=1)[:, None]
    b2 = np.sum(B ** 2, axis=1)[None, :]
    return np.sqrt(np.maximum(a2 + b2 - 2 * A @ B.T, 0))

def knn_radii(X, k):
    '''每个点到其第 k 近邻的距离(流形球半径)。'''
    D = pairwise_dist(X, X)
    np.fill_diagonal(D, np.inf)              # 排除自己
    D.sort(axis=1)
    return D[:, k - 1]

def precision_recall(real, gen, k=3):
    r_rad = knn_radii(real, k)               # 真实流形球半径
    g_rad = knn_radii(gen, k)                # 生成流形球半径
    # precision: 生成点是否落在任一真实球内
    prec = np.mean((pairwise_dist(gen, real) <= r_rad[None, :]).any(axis=1))
    # recall: 真实点是否落在任一生成球内
    rec = np.mean((pairwise_dist(real, gen) <= g_rad[None, :]).any(axis=1))
    return float(prec), float(rec)

d = 4
real = rng.standard_normal((300, d))
gen_good = rng.standard_normal((300, d))      # 匹配真实
p_good, r_good = precision_recall(real, gen_good)
print(f'好生成器:  precision={p_good:.2f}  recall={r_good:.2f}')
assert p_good > 0.7 and r_good > 0.7, '匹配真实分布 -> 保真和多样都高'
print('✅ precision-recall 从零实现: 匹配真实分布时两者都高')

## 6 · PR 的威力：分别诊断坍塌 vs 跑偏(FID 一个数做不到)

- **mode collapse**(方差收缩, 只覆盖真实一小块): **precision 高、recall 低**(生成的都真, 但漏模式)。
- **off-manifold**(均值偏移, 生成跑出真实分布): **precision 低**(生成的不像真的)。

这正是 model collapse 的指纹: **recall 崩而 precision 不变** —— 单看 FID 或保真度察觉不到。

In [ ]:
gen_narrow = rng.standard_normal((300, d)) * 0.25     # 方差收缩 = mode collapse
gen_shifted = rng.standard_normal((300, d)) + 3.0      # 均值偏移 = off-manifold

p_n, r_n = precision_recall(real, gen_narrow)
p_s, r_s = precision_recall(real, gen_shifted)
print(f'mode collapse(方差收缩): precision={p_n:.2f}  recall={r_n:.2f}  <- recall 崩!')
print(f'off-manifold(均值偏移) : precision={p_s:.2f}  recall={r_s:.2f}  <- precision 崩!')

assert r_n < 0.5, 'mode collapse -> recall 低(漏了大量真实模式)'
assert p_n > r_n, 'mode collapse -> precision 明显高于 recall(生成的都真但不全)'
assert p_s < 0.3, 'off-manifold -> precision 低(生成的不像真的)'
print('✅ PR 分别诊断: 坍塌=recall崩&precision不变(隐蔽!), 跑偏=precision崩 —— FID 单数看不出这区别')

---
## ✏️ 练习 1：FID（Fréchet 距离）

从零实现 `fid_compute(X, Y)`(用给定的 `matrix_sqrt`), 并验证: 自反性=0、对均值偏移与方差变化都敏感、对称(FID(X,Y)≈FID(Y,X))。

In [ ]:
def fid_compute(X, Y):
    # TODO: mu1,mu2 = 均值; s1,s2 = np.cov(., rowvar=False);
    #       FID = ||mu1-mu2||^2 + trace(s1 + s2 - 2*matrix_sqrt(s1@s2))
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
rng_t = np.random.default_rng(1)
X = rng_t.standard_normal((500, 6))
Y = rng_t.standard_normal((500, 6)) + 0.5
assert abs(fid_compute(X, X)) < 1e-6, 'FID(X,X)=0'
assert fid_compute(X, Y) > 0, '不同分布 FID>0'
# 对称性
assert abs(fid_compute(X, Y) - fid_compute(Y, X)) < 1e-6, 'FID 对称'
# 偏移越大 FID 越大
Y2 = rng_t.standard_normal((500, 6)) + 2.0
assert fid_compute(X, Y2) > fid_compute(X, Y), '均值偏移越大 FID 越大'
print(f'FID(X,X)={fid_compute(X,X):.4f}, FID(X,Y+0.5)={fid_compute(X,Y):.3f}, FID(X,Y+2)={fid_compute(X,Y2):.3f}')
print('✅ 练习 1 通过: FID 自反、对称、对分布差异敏感')

## ✏️ 练习 2：Inception Score（含 KL）

从零实现 `is_compute(probs)` 与 `kl_div(p, q)`。验证: 置信+多样 IS 高, 不置信或无多样 IS≈1。

In [ ]:
def kl_div(p, q, eps=1e-12):
    # TODO: sum(p * (log(p+eps) - log(q+eps)))  (单个分布 p 相对 q)
    raise NotImplementedError

def is_compute(probs):
    # TODO: p_y = probs.mean(0); 对每行算 kl_div(probs[i], p_y); 返回 exp(平均 KL)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# KL 基本性质
pp = np.array([0.7, 0.2, 0.1]); qq = np.array([0.5, 0.3, 0.2])
assert kl_div(pp, pp) < 1e-9, 'KL(p||p)=0'
assert kl_div(pp, qq) > 0, 'KL>=0, 不同分布>0'
C, N = 4, 800
lab = np.random.default_rng(2).integers(0, C, N)
conf_div = np.full((N, C), 0.02); conf_div[np.arange(N), lab] = 0.94
flat = np.full((N, C), 1.0 / C)
assert is_compute(conf_div) > 2.0, '置信+多样 IS 高'
assert abs(is_compute(flat) - 1.0) < 0.05, '不置信 IS≈1'
print(f'IS(置信+多样)={is_compute(conf_div):.3f}, IS(不置信)={is_compute(flat):.3f}')
print('✅ 练习 2 通过: IS 与 KL 正确')

## ✏️ 练习 3：precision-recall

实现 `pr_compute(real, gen, k)`(用给定的 `knn_radii`/`pairwise_dist`), 并用它**区分** mode collapse(recall低) 与 off-manifold(precision低)。

In [ ]:
def pr_compute(real, gen, k=3):
    # TODO: r_rad=knn_radii(real,k); g_rad=knn_radii(gen,k);
    #       precision = 生成点落在任一真实球内的比例
    #       recall    = 真实点落在任一生成球内的比例
    #       返回 (precision, recall)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
rng_t = np.random.default_rng(3)
R = rng_t.standard_normal((250, 4))
G_collapse = rng_t.standard_normal((250, 4)) * 0.25   # mode collapse
G_off = rng_t.standard_normal((250, 4)) + 3.0         # off-manifold
pc, rc = pr_compute(R, G_collapse)
po, ro = pr_compute(R, G_off)
print(f'mode collapse: P={pc:.2f} R={rc:.2f}')
print(f'off-manifold : P={po:.2f} R={ro:.2f}')
assert rc < pc, 'mode collapse: recall < precision'
assert po < 0.3, 'off-manifold: precision 低'
print('✅ 练习 3 通过: PR 区分坍塌(recall低)与跑偏(precision低)')

## ✏️ 练习 4：多样性(熵)与权衡

实现 `diversity_vs_threshold(samples, qualities, thresholds)`：对每个质量阈值过滤后, 返回 `(平均质量列表, 多样性列表)`(多样性用样本位置的标准差)。验证质量↑则多样性↓(保真-多样权衡)。

In [ ]:
def diversity_vs_threshold(positions, qualities, thresholds):
    # TODO: 对每个 th, keep = qualities>=th; 记录 qualities[keep].mean() 和 positions[keep].std()
    #       返回 (avg_quality_list, diversity_list)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
rng_t = np.random.default_rng(4)
M = 2000
pos = rng_t.uniform(-3, 3, M)
qual = np.exp(-0.3 * pos ** 2) + 0.1 * rng_t.standard_normal(M)   # 中心质量高
ths = [-0.5, 0.0, 0.5, 0.9]
aq, dv = diversity_vs_threshold(pos, qual, ths)
for t, a, d in zip(ths, aq, dv):
    print(f'阈值{t:+.1f}: 平均质量={a:.3f} 多样性={d:.3f}')
assert aq[-1] > aq[0], '过滤越严平均质量越高(precision↑)'
assert dv[-1] < dv[0], '过滤越严多样性越低(recall↓)'
print('✅ 练习 4 通过: 保真-多样权衡 —— 没有免费午餐')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def fid_compute(X, Y):
    mu1, mu2 = X.mean(0), Y.mean(0)
    s1 = np.cov(X, rowvar=False); s2 = np.cov(Y, rowvar=False)
    diff = mu1 - mu2
    return float(diff @ diff + np.trace(s1 + s2 - 2 * matrix_sqrt(s1 @ s2)))

In [ ]:
# 练习 2 参考答案
def kl_div(p, q, eps=1e-12):
    p = np.asarray(p); q = np.asarray(q)
    return float(np.sum(p * (np.log(p + eps) - np.log(q + eps))))

def is_compute(probs):
    p_y = probs.mean(0)
    kls = [kl_div(probs[i], p_y) for i in range(len(probs))]
    return float(np.exp(np.mean(kls)))

In [ ]:
# 练习 3 参考答案
def pr_compute(real, gen, k=3):
    r_rad = knn_radii(real, k); g_rad = knn_radii(gen, k)
    prec = np.mean((pairwise_dist(gen, real) <= r_rad[None, :]).any(axis=1))
    rec = np.mean((pairwise_dist(real, gen) <= g_rad[None, :]).any(axis=1))
    return float(prec), float(rec)

In [ ]:
# 练习 4 参考答案
def diversity_vs_threshold(positions, qualities, thresholds):
    aq, dv = [], []
    for th in thresholds:
        keep = qualities >= th
        aq.append(float(qualities[keep].mean()))
        dv.append(float(positions[keep].std()))
    return aq, dv

---
## 🧪 真实数据胶囊：真实图像 patch 特征上的 FID

用真实图像(尝试 matplotlib 自带的 **Grace Hopper** 真实照片, 失败回退到结构化真实数值)提取 patch 特征, 对比「真实 patch」与「加噪/模糊 patch」的 FID。真实图像特征上 FID 同样有效。

In [ ]:
def load_real_image_patches(patch=8, n=400, seed=0):
    '''尝试 matplotlib 的真实照片 -> 灰度 patch 特征; 失败回退到结构化数据。'''
    rng = np.random.default_rng(seed)
    try:
        import matplotlib.cbook as cbook, matplotlib.image as mpimg
        with cbook.get_sample_data('grace_hopper.jpg') as f:
            img = mpimg.imread(f)
        gray = img.mean(axis=2) / 255.0 if img.ndim == 3 else img / 255.0
        H, W = gray.shape
        feats = []
        for _ in range(n):
            i = rng.integers(0, H - patch); j = rng.integers(0, W - patch)
            feats.append(gray[i:i + patch, j:j + patch].ravel())
        return np.array(feats), 'matplotlib Grace Hopper(真实照片)'
    except Exception:
        # 回退: 自然图像 patch 的真实统计(相邻像素强相关 -> 低频为主)
        base = np.linspace(0, 1, patch)
        proto = np.add.outer(base, base) / 2.0           # 平滑梯度(自然图像低频特征)
        feats = proto.ravel()[None, :] + 0.05 * rng.standard_normal((n, patch * patch))
        return feats, '内置自然图像统计回退(平滑 patch)'

real_feats, src = load_real_image_patches(patch=8, n=400)
print(f'数据来源: {src}, 特征 {real_feats.shape}')
real_feats = (real_feats - real_feats.mean(0)) / (real_feats.std(0) + 1e-8)

rng = np.random.default_rng(0)
# 退化的「生成」: 在真实 patch 上加噪 / 收缩方差(模拟坏生成器)
noisy = real_feats + 0.8 * rng.standard_normal(real_feats.shape)   # 加噪(保真↓)
collapsed = real_feats * 0.3                                        # 方差收缩(多样↓)
resample_idx = rng.permutation(len(real_feats))
resample = real_feats[resample_idx]                                 # 同分布重排(应≈0)

print(f'FID(真实, 同分布重排) = {fid(real_feats, resample):.4f}  (应≈0)')
print(f'FID(真实, 加噪)       = {fid(real_feats, noisy):.3f}')
print(f'FID(真实, 方差收缩)   = {fid(real_feats, collapsed):.3f}')
assert fid(real_feats, resample) < 1e-6, '同数据重排 FID≈0'
assert fid(real_feats, noisy) > 1.0, '加噪推高 FID'
assert fid(real_feats, collapsed) > 1.0, '方差收缩推高 FID'
print('✅ 真实图像 patch 特征上 FID 有效: 加噪/坍塌都被检出, 同分布≈0')

**🧪 胶囊练习**：实现 `fid_self(feats)`：返回 `FID(feats, feats)`(应为 0, 验证自反性)。

In [ ]:
def fid_self(feats):
    # TODO: 返回 fid(feats, feats)
    raise NotImplementedError

In [ ]:
# 自测
v = fid_self(real_feats)
assert abs(v) < 1e-6, 'FID 自反性: FID(X,X)=0'
print(f'FID(真实, 真实) = {v:.2e} (自反性验证)')
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def fid_self(feats):
    return fid(feats, feats)

### 小结
- 生成评测的根本: **保真(单样本像不像真) 与 多样(覆不覆盖真实全部模式) 是两个独立维度**; 单一标量会把它们混在一起。
- **FID** = ||μ1-μ2||²(保真) + Tr协方差差(多样/结构); 对保真↓和多样↓都敏感; 但有高斯假设、有偏(须同样本数比较)。
- **IS** = exp(E KL(p(y|x)||p(y))); 奖励置信+均匀, **不需真实图但易钻空子**(Goodhart), 已大体被 FID/PR 取代。
- **precision-recall** 用 k-NN 流形把保真(precision)和多样(recall)**掰成两个数**; **model collapse 的指纹 = recall 崩而 precision 不变**。
- **保真-多样权衡**(温度/truncation/过滤)是真实存在的; 没有银弹 —— 多指标交叉 + 人评兜底; 对任何度量都先问「它量什么、能被怎样骗」。

🎉 **全课完结**: 从双下降、grokking/涌现, 到数据流水线、合成数据坍塌, 再到生成评测 —— 同一条纪律贯穿始终: **面对反常先归类、用最小模型验证机制、对任何度量都先问它在量什么。**